# Audio gate — presentation figures

**Purpose:** Two vector-PDF figures that explain the main audio design decisions using saved experiment results. The notebook does not run inference or retrain any model.

**Scope:** MammAlps Benchmark II, event-disjoint **test** set of 207 event-centered 2-second audio windows. Decision thresholds and early-exit policies were chosen using validation data. These results measure an audio gate, not end-to-end activity recognition or continuous deployment.

**Figures:**
1. **Adaptive PANNs:** accuracy–compute trade-off and the balanced policy's observed exit depths.
2. **Choosing the gate:** animal recall and background rejection for PANNs, MFCC + LR, and MFCC + RF.

Both figures are exported as **vector PDF** files to `Audio/results/presentation_figures/`. The figure text and plotted elements remain crisp when scaled in Overleaf.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

SPLIT = "test"

for location in (Path.cwd(), *Path.cwd().parents):
    if (location / "Audio" / "results").is_dir():
        ROOT = location
        break
else:
    raise FileNotFoundError("Project root not found: expected an Audio/results directory.")

RESULTS = ROOT / "Audio" / "results"
OUT = RESULTS / "presentation_figures"
OUT.mkdir(parents=True, exist_ok=True)

BLUE = "#2563EB"
TEAL = "#0F766E"
GRAY = "#64748B"
LIGHT = "#94A3B8"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12.5,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#CBD5E1",
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "pdf.fonttype": 42,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def load_csv(relative):
    path = RESULTS / relative
    if not path.is_file():
        raise FileNotFoundError(f"Saved results not found: {path}")
    return pd.read_csv(path)

def export(fig, name):
    path = OUT / f"{name}.pdf"
    fig.savefig(path, format="pdf", bbox_inches="tight", pad_inches=0.18)
    display(fig)
    plt.close(fig)
    print(f"PDF: {path.relative_to(ROOT)}")

panns_models = load_csv("panns_audio_all_models_comparison_2p0s.csv")
panns_policies = load_csv("panns_audio_policy_flops_2p0s.csv")
light_metrics = load_csv("mfcc_gate/mfcc_lr_rf_metrics_val_test.csv")

print("Dataset:", SPLIT, "| project:", ROOT)

## Step 1 — Can the large audio model adapt its computation?

The first design used pretrained **PANNs Cnn14**, with binary animal/background heads after Blocks 2, 4, and 6. The shared backbone allows a confident sample to stop early; an uncertain sample continues through deeper blocks.

**Question answered by Figure 1:** Does this conditional depth actually improve the accuracy–computation trade-off compared with always evaluating a fixed number of blocks?

**How to read it:** On the left, the preferable direction is **up and left** (higher accuracy, fewer operations). Squares are fixed-depth baselines; circles are early-exit policies. The right panel shows how frequently the **balanced** policy used each exit. PANNs operation counts use the same multiply–accumulate convention across configurations (approximately two FLOPs per MAC).

In [ ]:
compare = panns_models.loc[
    panns_models["split"].eq(SPLIT) & panns_models["type"].isin(["static", "adaptive"])
].copy()
policy = panns_policies.loc[
    panns_policies["split"].eq(SPLIT) & panns_policies["operating_point"].eq("balanced")
]
if len(compare) != 6 or len(policy) != 1:
    raise ValueError("Expected six static/adaptive PANNs points and one balanced-policy result.")
bal = policy.iloc[0]

fig = plt.figure(figsize=(12.5, 4.55), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.85, 1.0], wspace=0.14)
ax = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

static = compare.loc[compare.type.eq("static")]
adaptive = compare.loc[compare.type.eq("adaptive")]
ax.scatter(static.gflops_est, static.accuracy * 100, marker="s", s=104,
           facecolor=GRAY, edgecolor="white", linewidth=1.4, zorder=4)
other = adaptive.loc[adaptive.model.ne("balanced")]
ax.scatter(other.gflops_est, other.accuracy * 100, marker="o", s=118,
           facecolor=TEAL, edgecolor="white", linewidth=1.4, zorder=4)
balanced = adaptive.loc[adaptive.model.eq("balanced")].iloc[0]
ax.scatter([balanced.gflops_est], [balanced.accuracy * 100], marker="*", s=270,
           facecolor=BLUE, edgecolor="white", linewidth=1.0, zorder=6)

labels = {
    "Exit 1 / block 2": ("Fixed B2", -1, -21),
    "Exit 2 / block 4": ("Fixed B4", 7, -20),
    "Final / block 6": ("Fixed B6", -2, 10),
    "aggressive": ("Aggressive", 10, 7),
    "balanced": ("Balanced", 10, 12),
    "conservative": ("Conservative", 9, -17),
}
for _, r in compare.iterrows():
    name, dx, dy = labels[r.model]
    ax.annotate(name, (r.gflops_est, r.accuracy * 100), xytext=(dx, dy),
                textcoords="offset points", fontsize=9.4,
                weight="bold" if name == "Balanced" else "normal",
                color=BLUE if name == "Balanced" else DARK)

ax.legend(handles=[
    Line2D([0], [0], marker="s", color="none", markerfacecolor=GRAY,
           markeredgecolor="white", markersize=9, label="Fixed depth"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor=TEAL,
           markeredgecolor="white", markersize=9, label="Early exit"),
    Line2D([0], [0], marker="*", color="none", markerfacecolor=BLUE,
           markeredgecolor="white", markersize=12, label="Balanced"),
], loc="upper right", frameon=False, fontsize=9)

ax.set(xlabel="Estimated compute (GFLOPs / audio window)",
       ylabel="Binary classification accuracy (%)", title="Accuracy versus compute")
ax.set_xlim(2.35, 9.06)
ax.set_ylim(78.2, 98.0)
ax.grid(alpha=0.15, lw=0.8)
ax.set_axisbelow(True)

exit_names = ["Block 2", "Block 4", "Block 6"]
fractions = np.array([bal.exit1_rate, bal.exit2_rate, bal.final_rate])
colors = ["#93C5FD", BLUE, "#334155"]
positions = np.arange(3)
bars = ax2.bar(positions, fractions * 100, color=colors, width=0.66,
               edgecolor="white", linewidth=1.0)
for i, (bar, value) in enumerate(zip(bars, fractions)):
    ax2.annotate(f"{value*100:.1f}%", (bar.get_x()+bar.get_width()/2, value*100),
                 xytext=(0, 6), textcoords="offset points", ha="center", fontsize=11,
                 weight="bold" if i < 2 else "normal", color=DARK)
ax2.set_xticks(positions, exit_names)
ax2.set(ylabel="Share of test windows (%)", title="Balanced policy: actual exit depth")
ax2.set_ylim(0, 63)
ax2.grid(axis="y", alpha=0.15)
ax2.set_axisbelow(True)
ax2.spines["left"].set_visible(False)

export(fig, "01_panns_adaptive_tradeoff")

block4 = static.loc[static.model.eq("Exit 2 / block 4")].iloc[0]
saving = 100 * (1 - balanced.gflops_est / block4.gflops_est)
print(f"Balanced: {balanced.accuracy*100:.2f}% accuracy, {balanced.gflops_est:.3f} GFLOPs/window")
print(f"Fixed block 4: {block4.accuracy*100:.2f}% accuracy, {block4.gflops_est:.3f} GFLOPs/window")
print(f"Compute reduction relative to fixed block 4: {saving:.1f}%")
n_windows = int(sum(float(bal[key]) for key in ["tn", "fp", "fn", "tp"]))
print("Balanced exit counts:", [round(float(rate) * n_windows) for rate in fractions])

**What this step establishes.** Balanced PANNs achieved about **93.2% test accuracy at 4.26 GFLOPs per window**, compared with **92.8% at 5.66 GFLOPs** for the fixed Block 4 baseline: approximately **24.7% less estimated compute** for a similar observed accuracy. Almost every test sample stopped at Block 2 or Block 4. The small accuracy difference on 207 windows should not be presented as a statistically established improvement.

**Transition to the next step:** Early exits reduce PANNs computation, but this is still a relatively large acoustic model for an always-on binary gate. The next question is whether a lightweight classifier retains adequate animal recall.

## Step 2 — Is a smaller audio gate sufficient?

**MFCC + Logistic Regression** and **MFCC + Random Forest** were trained for the same animal/background task, using 40 MFCC summary features (20 means and 20 standard deviations). Their decision thresholds were calibrated on validation data to prioritize animal recall.

**Question answered by Figure 2:** How many animal windows are detected, and how many background windows are rejected, compared with the balanced PANNs gate?

All models use the same **123 animal** and **84 background** test windows. Animal recall and background rejection are intentionally shown on separate axes because they represent different failure costs. The x-axes use different ranges; comparisons should use labeled percentages, not visual bar lengths.

In [ ]:
selected = panns_policies.loc[
    panns_policies["split"].eq(SPLIT) & panns_policies["operating_point"].eq("balanced")
].iloc[0]
metrics = light_metrics.loc[
    light_metrics["split"].eq(SPLIT) & light_metrics["model"].isin(["MFCC + LR", "MFCC + RF"])
].copy()
if len(metrics) != 2:
    raise ValueError("Expected test metrics for MFCC + LR and MFCC + RF.")

rows = [{"model": "PANNs (balanced)", "animal_recall": float(selected.animal_recall),
         "background_rejection": float(selected.background_rejection),
         "fn": int(selected.fn), "fp": int(selected.fp)}]
for _, r in metrics.iterrows():
    rows.append({"model": r.model, "animal_recall": float(r.animal_recall),
                 "background_rejection": float(r.background_rejection),
                 "fn": int(r.fn), "fp": int(r.fp)})
comparison = pd.DataFrame(rows).set_index("model").loc[
    ["PANNs (balanced)", "MFCC + LR", "MFCC + RF"]
].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.1), sharey=True,
                         layout="constrained", gridspec_kw={"wspace": 0.15})
y = np.arange(len(comparison))[::-1]
colors = [BLUE, TEAL, LIGHT]
for ax, metric, title, min_axis, counts, total in [
    (axes[0], "animal_recall", "Animal detected (higher is better)", 90, "fn", 123),
    (axes[1], "background_rejection", "Background rejected (higher is better)", 65, "fp", 84),
]:
    for i, r in comparison.iterrows():
        score = float(r[metric]) * 100
        ax.hlines(y[i], min_axis, score, color=colors[i], lw=3, alpha=0.7)
        ax.scatter([score], [y[i]], color=colors[i], s=125, edgecolor="white", lw=1.1, zorder=4)
        count = int(r[counts])
        text = f"{score:.1f}%   ({count} {'missed' if counts == 'fn' else 'false alarms'})"
        ax.annotate(text, (score, y[i]), xytext=(7, 0), textcoords="offset points",
                    va="center", fontsize=10, color=DARK)
    ax.set(xlim=(min_axis, 108.5), ylim=(-0.55, 2.55),
           xlabel="Share of ground-truth windows (%)", title=title)
    ax.grid(axis="x", color="#CBD5E1", lw=0.75, alpha=0.45)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)

axes[0].set_yticks(y, comparison.model.tolist())
axes[1].tick_params(labelleft=False)
axes[0].set_xticks([90, 95, 100])
axes[1].set_xticks([65, 75, 85, 95])
export(fig, "02_audio_gate_selection")

display(comparison)

**What this step establishes.** On the test set, PANNs balanced and MFCC + LR each missed **2 of 123 animal windows** (98.4% animal recall). PANNs rejected **72 of 84** background windows (85.7%), while LR rejected **69 of 84** (82.1%): LR therefore triggered video **three extra times** on background. MFCC + RF missed four animal windows and had lower background rejection than LR.

A separate CPU benchmark measured a median **5.72 ms per 2-second window** for MFCC + LR, including cached audio reading, resampling, feature extraction, and classifier inference. This timing supports the lightweight design but is **not directly comparable** with the PANNs profiled GFLOPs, and does not measure camera energy consumption.

**Decision supported by the experiments:** PANNs is valuable as a shared-weight adaptive neural architecture; MFCC + LR is a reasonable candidate for a cheap always-on gate. The final architecture decision depends on measured conditional audio–video pipeline cost.

## Suggested audio presentation sequence

1. **Problem and decision criterion (no plot):** The camera should not classify video on every interval. Missing an animal is more costly than an unnecessary video activation, so animal recall takes priority.
2. **Adaptive model design (brief block diagram):** PANNs has heads at Blocks 2, 4, and 6. A confidence margin determines whether an input exits or continues. The heads share the same backbone.
3. **Proof of adaptive computation (Figure 1):** Show the accuracy–FLOPs plot, then point to the observed exit depths. The key comparison is balanced PANNs versus fixed Block 4, not just versus the full network.
4. **Simplifying the always-on stage (Figure 2):** Ask whether the gate needs a large neural network. The lightweight LR classifier misses the same number of animal windows but produces three additional false activations on this test split.
5. **System-level follow-up (separate pipeline results):** Evaluate video-only, PANNs + adaptive video, and LR + adaptive video on matched raw-video windows, including gate costs and actual conditional video activations. The current two audio figures cannot establish end-to-end activity accuracy or battery life.

**Backup material:** Pruning/INT8 results and the CPU quantization failure are useful in questions or a limitations slide, but they do not need separate figures in the main audio narrative. The confusion matrices repeat counts already shown in Figure 2. A standalone latency chart can be omitted if the 5.72 ms LR timing is stated clearly in the slide caption or narration.

**Overleaf:** `\includegraphics[width=\linewidth]{figures/01_panns_adaptive_tradeoff.pdf}` and `\includegraphics[width=\linewidth]{figures/02_audio_gate_selection.pdf}`. The figures contain vector graphics and embedded text.